In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# 检查环境变量
print("LANGCHAIN_TRACING_V2:", os.getenv("LANGCHAIN_TRACING_V2"))
print("LANGCHAIN_API_KEY:", os.getenv("LANGCHAIN_API_KEY")[:5] + "...")  # 只显示前5位

LANGCHAIN_TRACING_V2: true
LANGCHAIN_API_KEY: lsv2_...


In [2]:
from tools import tools
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory

# 复用 Day8 的 LLM 和 memory 配置
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)  # 你的配置
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
react_prompt = ChatPromptTemplate.from_template("""
你是一个乐于助人的助手。你可以使用以下工具：

{tools}

对话历史：
{chat_history}

请严格遵循以下格式：

Question: 用户的问题
Thought: 你应该思考要做什么
Action: 要采取的行动，必须是 [{tool_names}] 中的一个
Action Input: 行动的输入参数
Observation: 行动的结果
... (可以重复多次 Thought/Action/Action Input/Observation)
Thought: 我现在知道最终答案了
Final Answer: 对原始问题的最终答案

开始！

Question: {input}
Thought: {agent_scratchpad}
""")  # 你的模板

agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, memory=memory, verbose=True)

# 运行一个测试
result = agent_executor.invoke({"input": "方闻馆的开放时间"})
print(result["output"])

C:\Users\naizz\ai_first_call\tools.py:31: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


向量库加载成功


C:\Users\naizz\AppData\Local\Temp\ipykernel_21036\3227483685.py:14: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)




> Entering new AgentExecutor chain...
Thought: 用户询问的是图书馆的开放时间，这属于图书馆相关规则的问题，应该使用rag_search_tool进行检索。

Action: rag_search_tool
Action Input: 方闻馆的开放时间【借阅规则】（2025年4月修订）
─────────────────────────────────

一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册        3册
全日制本科生          100册       30册        3册
全日制研究生          100册       30册        3册
非全日制研究生        30册        3册         1册
企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册         1册         1册
继续教育读者          5册         1册         1册

注：已退休但继续承担教学科研任务的教师，凭所在学院证明可申请保留100册借书量。

浙江大学图书馆使用指南
═══════════════════════════════════════════════════════

【图书馆概况】
浙江大学图书馆是浙江省规模最大、设备最齐全的高等学校图书馆之一，由紫金港校区的主馆、基础馆、农医馆、古籍馆、方闻馆和玉泉分馆、西溪分馆、华家池分馆组成。

【开放时间】（常规时间）

一、紫金港主馆
─────────────────────────────────
周一至周日：8:30-22:30
（注：节假日可能有调整，如除夕调整为8:30-17:00）

二、基础馆、农

In [4]:
# 测试一个包含工具调用的问题
result_1 = agent_executor.invoke({"input": "续借规则是什么？"})
print("答案：", result_1["output"])



> Entering new AgentExecutor chain...
Thought: 对话历史中已经包含了关于续借规则的完整回答，无需再次查询工具。
Final Answer: 方闻馆的续借规则如下：

1. 读者可在到期前进行网上续借
2. 在999天最长可外借天数内续借次数不限
3. 已被预约图书不能续借
4. 凡有逾期图书，所借所有图书不得再办理续借手续

此外，普通流通图书外借期限为40天，短期外借图书外借期限为14天（不可预约，可续借1次）。如有逾期图书，需按照每日每本0.05元的罚金缴纳罚款。

> Finished chain.
答案： 方闻馆的续借规则如下：

1. 读者可在到期前进行网上续借
2. 在999天最长可外借天数内续借次数不限
3. 已被预约图书不能续借
4. 凡有逾期图书，所借所有图书不得再办理续借手续

此外，普通流通图书外借期限为40天，短期外借图书外借期限为14天（不可预约，可续借1次）。如有逾期图书，需按照每日每本0.05元的罚金缴纳罚款。


In [5]:
# 测试一个包含工具调用的问题
result_2 = agent_executor.invoke({"input": "芸悦读平台规则是什么？"})
print("答案：", result_2["output"])



> Entering new AgentExecutor chain...
Thought: 用户询问的是关于"芸悦读"平台的规则，这属于图书馆相关规则查询，应该使用rag_search_tool进行检索。

Action: rag_search_tool
Action Input: 芸悦读平台规则【入馆须知】
─────────────────────────────────
1. 全日制学生和事业编制教工校园一卡通借书功能自动开通
2. 全日制学生拿到一卡通时自动开通
3. 事业编制教工在人事处报到后1-2周内自动开通
4. 未自动开通的其他人员需携带相关证明，并交纳押金开通借书功能
5. 开通后需自行激活借书权限

【借阅规则】（2025年4月修订）
─────────────────────────────────

六、借阅范围
─────────────────────────────────
可借：书目查询中显示"普通外借"的图书
仅供室内阅览：特藏书（善本书、线装书、浙大文库藏书等）、参考工具书等

【芸悦读平台】
─────────────────────────────────
1. 借阅数上限：15本
2. 还书专窗：各校区分馆设有芸悦读还书专窗
3. 暑假期间借阅期自动延长（根据校历调整）

【文明行为规范】
─────────────────────────────────
1. 不占座，不霸座，按时到场，根据预约的座位就座
2. 在图书馆内就座时，不应饮食、交谈或主观发出较大声响
3. 请勿在图书馆内嬉戏打闹
4. 将电子设备调至静音，使用键盘时注意对他人影响
5. 保持安静、整洁的学习环境

【联系方式】
─────────────────────────────────
馆长信箱：infolib@zju.edu.cn
留言板：http://libweb.zju.edu.cn/55982/list.htm
海宁国际校区图书馆：0571-87572288

═══════════════════════════════════════════════════════
信息来源：浙江大学图书馆官方网站（libweb.zju.edu.cn）

注：已退休但继续承担教学科研任务的教师，凭所在学院证明可申请保留100册借书量。

二、借阅期限

In [6]:
# 测试一个包含工具调用的问题
result_3 = agent_executor.invoke({"input": "方闻馆开放时间？今天北京天气？15+27？"})
print("答案：", result_3["output"])



> Entering new AgentExecutor chain...
Thought: 我需要分别处理三个问题：查询方闻馆开放时间、查询北京天气、计算15+27的结果。我将按顺序处理这些问题。

Action: rag_search_tool
Action Input: 方闻馆开放时间【借阅规则】（2025年4月修订）
─────────────────────────────────

一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册        3册
全日制本科生          100册       30册        3册
全日制研究生          100册       30册        3册
非全日制研究生        30册        3册         1册
企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册         1册         1册
继续教育读者          5册         1册         1册

注：已退休但继续承担教学科研任务的教师，凭所在学院证明可申请保留100册借书量。

浙江大学图书馆使用指南
═══════════════════════════════════════════════════════

【图书馆概况】
浙江大学图书馆是浙江省规模最大、设备最齐全的高等学校图书馆之一，由紫金港校区的主馆、基础馆、农医馆、古籍馆、方闻馆和玉泉分馆、西溪分馆、华家池分馆组成。

【开放时间】（常规时间）

一、紫金港主馆
─────────────────────────────────
周一至周日：8:30-22:30
（注：节假日可能有调整，如除夕调整为8:30-17:00）

二、基础馆、农医馆

In [7]:
result_4 = agent_executor.invoke({"input": "请告诉我现在北京和上海的气温"})
print("答案：", result_4["output"])



> Entering new AgentExecutor chain...
我需要查询北京和上海两个城市的实时天气信息。
Action: weather_tool
Action Input: 北京查询失败：city not found查询北京天气失败，我需要尝试其他方式获取天气信息。现在尝试查询上海天气。

Action: weather_tool  
Action Input: 上海  查询失败：city not found两个城市的天气查询都失败了，可能是天气服务暂时不可用。

Final Answer: 抱歉，我暂时无法获取北京和上海的实时天气信息。您可以稍后再试或通过其他天气平台查询。

> Finished chain.
答案： 抱歉，我暂时无法获取北京和上海的实时天气信息。您可以稍后再试或通过其他天气平台查询。


In [8]:
result_5 = agent_executor.invoke({"input": "请告诉我现在北京市和上海市的气温"})
print("答案：", result_5["output"])



> Entering new AgentExecutor chain...
Thought: 我需要查询北京市和上海市的实时天气信息，应该使用weather_tool分别查询两个城市的气温。

Action: weather_tool
Action Input: 北京查询失败：city not foundThought: 天气查询工具似乎无法识别"北京"这个城市名，我尝试使用"北京市"作为查询参数。

Action: weather_tool
Action Input: 北京市北京市 天气：多云，温度 24.43℃，湿度 21%Thought: 现在我需要查询上海市的气温，同样使用weather_tool工具。

Action: weather_tool
Action Input: 上海市上海市 天气：阴，多云，温度 17.33℃，湿度 42%Thought: 我现在知道最终答案了
Final Answer: 当前北京市和上海市的气温如下：
- 北京市：多云，温度24.43℃
- 上海市：阴，多云，温度17.33℃

> Finished chain.
答案： 当前北京市和上海市的气温如下：
- 北京市：多云，温度24.43℃
- 上海市：阴，多云，温度17.33℃


In [16]:
result_6 = agent_executor.invoke({"input": "请告诉我现在杭州和南昌的气温，你需要尝试在地名后面加市进行查询，例如成都市"})
print("答案：", result_6["output"])



> Entering new AgentExecutor chain...


INFO:httpx:HTTP Request: POST https://api.siliconflow.cn/v1/chat/completions "HTTP/1.1 200 OK"


Thought: 我需要查询杭州市和南昌市的实时天气信息
Action: weather_tool
Action Input: 杭州市杭州市 天气：多云，温度 13.05℃，湿度 69%

INFO:httpx:HTTP Request: POST https://api.siliconflow.cn/v1/chat/completions "HTTP/1.1 200 OK"


Thought: 我还需要查询南昌市的实时天气信息
Action: weather_tool
Action Input: 南昌市南昌市 天气：小雨，温度 17.38℃，湿度 82%

INFO:httpx:HTTP Request: POST https://api.siliconflow.cn/v1/chat/completions "HTTP/1.1 200 OK"


Thought: 我现在知道最终答案了
Final Answer: 当前杭州市和南昌市的气温如下：
- 杭州市：多云，温度13.05℃
- 南昌市：小雨，温度17.38℃

> Finished chain.
答案： 当前杭州市和南昌市的气温如下：
- 杭州市：多云，温度13.05℃
- 南昌市：小雨，温度17.38℃


In [10]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()

def contains_keywords(run, example):
    """检查答案是否包含 '方闻馆' 和 '开放时间' 关键字"""
    output = run.outputs.get("output", "")
    return "方闻馆" in output and "开放时间" in output

# 在 LangSmith 中注册评估器（需要上传数据集，高级功能）

In [12]:
result_7 = agent_executor.invoke({"input": "帮我订一张机票?"})
print("答案：", result_7["output"])



> Entering new AgentExecutor chain...
Thought: 我是一个专注于提供信息查询和简单计算的助手，不具备订票功能。
Final Answer: 抱歉，我无法帮您订机票。建议您通过航空公司官网或正规票务平台预订。

> Finished chain.
答案： 抱歉，我无法帮您订机票。建议您通过航空公司官网或正规票务平台预订。


In [15]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def rag_search_tool(query: str) -> str:
    logger.info(f"rag_search_tool 被调用，查询：{query}")
    result = _search_library(query)
    logger.info(f"返回结果长度：{len(result)}")
    return result

In [18]:
# 在调用 agent_executor 前后添加日志
logger.info(f"用户提问：{prompt}")
response = agent_executor.invoke({"input": prompt})
answer = response["output"]
logger.info(f"Agent 回答：{answer[:200]}...")  # 只记录前200字符避免日志过长

NameError: name 'prompt' is not defined